# Art Therapy Questionnaire Analysis for HCI Research

This notebook analyzes questionnaire responses from an art-therapy HCI study. It focuses on how drawing skills, participation, enjoyment, and four drawing-process steps relate to emotional outcomes.

**Run this notebook from the project environment** after installing `requirements-analysis.txt`. No external API is required.

## Data source

The notebook prefers the local `Dataset/art_data.xlsx` research workbook. If that file is unavailable, it automatically uses the publishable synthetic example at `Dataset/example/art_data.xlsx`. The required 16-column schema is documented in `Dataset/README.md`.

## Analysis workflow

| Section | Steps | Purpose |
|---|---:|---|
| Data validation | Step 1 | Locate the workbook, verify required columns and IDs, and audit multimodal data availability |
| Predictive modeling | Steps 2–3, 22 | Evaluate Elastic Net and histogram gradient boosting models with cross-validation |
| Group comparison | Steps 4–15 | Compare drawing-step ratings across three drawing-skill levels |
| Correlation analysis | Steps 16–21 | Estimate step-to-emotion associations within each skill group |
| Hierarchical regression | Step 23 | Test the incremental contribution of participation and drawing-step factors |
| Alignment analysis | Steps 24–25 | Examine whether participant preferences align with the steps associated with positive emotion |

## Output

Results are displayed inside the notebook as model metrics, statistical tables, and plots. The source workbook is never overwritten.

> These analyses are exploratory and observational. Associations should not be interpreted as causal effects or clinical conclusions, and participant-level data must not be published.


## Step 1: Locate and Validate the Dataset

Find the repository root, prefer the local research dataset over the synthetic example, load the workbook, and validate its required columns and participant IDs. Then audit image, text, and core numerical-data availability, including all eight exact availability combinations and analysis-specific sample sizes.

Availability definitions: an image is available only when `processed_painting_path` points to an existing file; text must be non-empty and not an `na`-style placeholder; core numerical data require valid values for the outcome, four drawing-process variables, and four control variables used in the main analyses.


In [ ]:
from pathlib import Path
import pandas as pd

def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        real_file = candidate / "Dataset" / "art_data.xlsx"
        example_file = candidate / "Dataset" / "example" / "art_data.xlsx"
        if real_file.is_file() or example_file.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find Dataset/art_data.xlsx or Dataset/example/art_data.xlsx. Start Jupyter from the repository root."
    )

REPO_ROOT = find_repo_root()
DATASET_DIR = REPO_ROOT / "Dataset"
REAL_DATA_FILE = DATASET_DIR / "art_data.xlsx"
EXAMPLE_DATA_FILE = DATASET_DIR / "example" / "art_data.xlsx"
DATA_FILE = REAL_DATA_FILE if REAL_DATA_FILE.is_file() else EXAMPLE_DATA_FILE
USING_EXAMPLE_DATA = DATA_FILE == EXAMPLE_DATA_FILE

REQUIRED_COLUMNS = [
    "student_id", "drawing_skills", "drawing_participation", "enjoy_drawing",
    "negative_release", "write_emotions", "imagine_composition", "examples",
    "color_expression", "others", "positive_emotion", "continue_participation",
    "level_difficulty", "painting_path", "processed_painting_path", "text",
]

df_raw = pd.read_excel(DATA_FILE)
df_raw.columns = [str(c).strip() for c in df_raw.columns]
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df_raw.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")
if df_raw["student_id"].duplicated().any():
    raise ValueError("student_id contains duplicate values. Inspect the dataset before continuing.")

# Use a conservative definition of numerical completeness for multimodal alignment.
CORE_NUMERIC_COLUMNS = [
    "positive_emotion",
    "write_emotions", "imagine_composition", "examples", "color_expression",
    "drawing_skills", "level_difficulty", "drawing_participation", "enjoy_drawing",
]

MISSING_TEXT_MARKERS = {"", "na", "n/a", "nan", "none", "null"}

def has_meaningful_text(value):
    if pd.isna(value):
        return False
    return str(value).strip().lower() not in MISSING_TEXT_MARKERS

def referenced_file_exists(value):
    if not has_meaningful_text(value):
        return False
    path = Path(str(value).strip())
    if not path.is_absolute():
        path = REPO_ROOT / path
    return path.is_file()

numeric_values = df_raw[CORE_NUMERIC_COLUMNS].apply(pd.to_numeric, errors="coerce")
availability = pd.DataFrame({
    "image": df_raw["processed_painting_path"].map(referenced_file_exists),
    "text": df_raw["text"].map(has_meaningful_text),
    "numeric": numeric_values.notna().all(axis=1),
})

availability_summary = pd.DataFrame({
    "data_type": ["Existing processed image", "Meaningful text", "Complete core numerical data", "All three aligned"],
    "rows": [
        int(availability["image"].sum()),
        int(availability["text"].sum()),
        int(availability["numeric"].sum()),
        int(availability.all(axis=1).sum()),
    ],
})

all_patterns = pd.MultiIndex.from_product(
    [[False, True], [False, True], [False, True]],
    names=["image", "text", "numeric"],
)
combination_summary = (
    availability.value_counts(sort=False)
    .reindex(all_patterns, fill_value=0)
    .rename("rows")
    .reset_index()
)
for column in ["image", "text", "numeric"]:
    combination_summary[column] = combination_summary[column].map({True: "Yes", False: "No"})
combination_summary = combination_summary.sort_values("rows", ascending=False).reset_index(drop=True)

analysis_sample_sizes = pd.DataFrame({
    "analysis_requirement": [
        "Numerical-only complete-case analysis",
        "Image-only feature extraction or labeling",
        "Text-only linguistic analysis",
        "Image-text semantic comparison",
        "Image-numerical analysis",
        "Text-numerical analysis",
        "Integrated image-text-numerical analysis",
    ],
    "available_rows": [
        int(availability["numeric"].sum()),
        int(availability["image"].sum()),
        int(availability["text"].sum()),
        int((availability["image"] & availability["text"]).sum()),
        int((availability["image"] & availability["numeric"]).sum()),
        int((availability["text"] & availability["numeric"]).sum()),
        int(availability.all(axis=1).sum()),
    ],
})

df = df_raw.copy()
print(f"Loaded {len(df_raw)} rows from {DATA_FILE}")
if USING_EXAMPLE_DATA:
    print("Using synthetic example data; results are for demonstration only.")

print("\nAvailability by data type:")
print(availability_summary.to_string(index=False))
print("\nAll exact availability combinations:")
print(combination_summary.to_string(index=False))
print("\nUsable sample size by analysis requirement:")
print(analysis_sample_sizes.to_string(index=False))

## Step 2: Build and Evaluate an Elastic Net Model

Prepare numeric predictors, handle missing values, fit an Elastic Net regression pipeline, and report five-fold cross-validation metrics for positive emotion.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_squared_error, make_scorer


df.columns = [c.strip() for c in df.columns]
df = df.replace({"na": np.nan, "NA": np.nan, "N/A": np.nan, "": np.nan})

#set up x and y
y_col = "positive_emotion"
drop_cols = ["student_id", "text", "continue_participation",'painting_path',
             'processed_painting_path','negative_release',
             'others']  # delete the unwanted columns.

# Convert non-text and non-path columns to numeric values; invalid values become NaN.
for c in df.columns:
    if c not in {"text", "painting_path", "processed_painting_path"}:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Drop rows with a missing outcome.
df = df.dropna(subset=[y_col])

y = df[y_col].astype(float)
X = df.drop(columns=[y_col] + drop_cols, errors="ignore")

# 4) Regression pipeline: median imputation, missingness indicators, scaling, and Elastic Net/Lasso.
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),  # Treat whether a value was missing as an additional feature.
    ("scaler", StandardScaler()),
    ("reg", ElasticNetCV(
        l1_ratio=[.1, .3, .5, .7, .9, 1.0],  # 1.0 is Lasso and applies the strongest variable selection.
        cv=5,
        random_state=42,
        max_iter=20000
    ))
])

# 5) Cross-validation evaluation.
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(
    model, X, y, cv=cv,
    scoring={
        "r2": "r2",
        "neg_mae": "neg_mean_absolute_error",
        "neg_rmse": rmse_scorer
    }
)

print("CV R2    :", scores["test_r2"].mean(), "+/-", scores["test_r2"].std())
print("CV MAE   :", -scores["test_neg_mae"].mean(), "+/-", scores["test_neg_mae"].std())
print("CV RMSE  :", -scores["test_neg_rmse"].mean(), "+/-", scores["test_neg_rmse"].std())

model.fit(X, y)
reg = model.named_steps["reg"]


## Step 3: Inspect Elastic Net Coefficients

Recover the transformed feature names and rank the fitted coefficients to identify the strongest positive and negative predictors.


In [ ]:
imputer = model.named_steps["imputer"]
reg = model.named_steps["reg"]

# Retrieve the actual feature names produced by the imputer transformation.
feat_names = imputer.get_feature_names_out(X.columns)
coef = reg.coef_

print("len(feat_names) =", len(feat_names))
print("len(coef)       =", len(coef))

coef_df = pd.DataFrame({
    "feature": feat_names,
    "coef": coef
}).sort_values("coef", ascending=False)

print("\nTop positive predictors:")
print(coef_df.head(10).to_string(index=False))

print("\nTop negative predictors:")
print(coef_df.tail(10).to_string(index=False))

## Step 4: Select Variables for Group Analysis

Create a compact analysis table containing drawing-skill level, the four drawing-step ratings, and the two emotion outcomes.


In [ ]:
use = df_raw[[
    "drawing_skills", "write_emotions", "imagine_composition", "examples",
    "color_expression", "negative_release", "positive_emotion",
]].copy()
use.columns = ["level", "write", "imagine", "ref", "color",'neg','pos']
use.head()


## Step 5: Clean the Group-Analysis Table

Keep valid drawing-skill groups, convert the remaining fields to numeric values, and report the resulting table size.


In [ ]:
use = use.dropna(subset=["level"])
use = use[use["level"].isin([1, 2, 3])]

for c in use.columns[1:]:
    use[c] = pd.to_numeric(use[c], errors="coerce")

use.shape


## Step 6: Summarize Mean Scores by Skill Level

Calculate the average drawing-step and emotion scores for each drawing-skill level.


In [ ]:
use.groupby(use.columns[0])[use.columns[1:]].mean().round(2)


## Step 7: Count Participants in Each Skill Level

Display the number of usable participants in drawing-skill levels 1, 2, and 3.


In [ ]:
use[use.columns[0]].value_counts().sort_index()


## Step 8: Run Kruskal-Wallis Group Tests

Test whether each of the four drawing-step ratings differs across the three drawing-skill levels.


In [ ]:
from scipy.stats import kruskal

level = use.columns[0]
write, imagine, ref, color = use.columns[1], use.columns[2], use.columns[3], use.columns[4]

# 1) Writing down emotions.
p_write = kruskal(
    use.loc[use[level]==1, write].dropna(),
    use.loc[use[level]==2, write].dropna(),
    use.loc[use[level]==3, write].dropna()
).pvalue

# 2) Imagining the composition.
p_imagine = kruskal(
    use.loc[use[level]==1, imagine].dropna(),
    use.loc[use[level]==2, imagine].dropna(),
    use.loc[use[level]==3, imagine].dropna()
).pvalue

# 3) Using reference images.
p_ref = kruskal(
    use.loc[use[level]==1, ref].dropna(),
    use.loc[use[level]==2, ref].dropna(),
    use.loc[use[level]==3, ref].dropna()
).pvalue

# 4) Expressing emotions with color.
p_color = kruskal(
    use.loc[use[level]==1, color].dropna(),
    use.loc[use[level]==2, color].dropna(),
    use.loc[use[level]==3, color].dropna()
).pvalue

p_write, p_imagine, p_ref, p_color


## Step 9: Compare Significance and Effect Sizes

Calculate Kruskal-Wallis statistics and epsilon-squared effect sizes, then visualize the evidence and magnitude of group differences.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kruskal

# Assume that the `use` analysis table was prepared above.
level = use.columns[0]
write, imagine, ref, color = use.columns[1], use.columns[2], use.columns[3], use.columns[4]

steps = [
    ("wring_down_emotions", write),
    ("imagine_composition", imagine),
    ("reference_to_the_picture", ref),
    ("color_reference", color),
]

def kruskal_p_and_eps2(df, level_col, value_col, groups=(1,2,3)):
    # Extract the observations for each skill group.
    arrays = []
    ns = []
    for g in groups:
        a = df.loc[df[level_col] == g, value_col].dropna().values
        arrays.append(a)
        ns.append(len(a))

    # Kruskal-Wallis
    res = kruskal(*arrays, nan_policy='omit')
    H = res.statistic
    p = res.pvalue

    # Epsilon-squared effect size, using a common Kruskal-Wallis approximation.
    # eps2 = (H - k + 1) / (n - k)
    k = len(groups)
    n = int(np.sum(ns))
    eps2 = (H - k + 1) / (n - k) if (n - k) > 0 else np.nan
    eps2 = max(0.0, float(eps2)) if np.isfinite(eps2) else np.nan  # Guard against negative estimates.

    return p, H, eps2, ns

rows = []
for name, col in steps:
    p, H, eps2, ns = kruskal_p_and_eps2(use, level, col, groups=(1,2,3))
    rows.append({"step": name, "p": p, "H": H, "eps2": eps2, "n_by_group": ns})

res_df = pd.DataFrame(rows)

# --- Figure 1: Compare the strength of the p-values. ---
# Use -log10(p) for readability and clip exact zeros to a safe lower bound.
p_safe = res_df["p"].clip(lower=1e-300)
res_df["neglog10p"] = -np.log10(p_safe)

plt.figure(figsize=(8, 4.5))
plt.bar(res_df["step"], res_df["neglog10p"])
plt.ylabel(r"$-\log_{10}(p)$  (larger = stronger difference evidence)")
plt.title("Comparison of significance strength across four steps by level (Kruskal–Wallis)")
plt.xticks(rotation=0)

# Add reference lines for the p = 0.05, 0.01, and 0.001 thresholds.
for p_thr in [0.05, 0.01, 0.001]:
    y = -np.log10(p_thr)
    plt.axhline(y, linestyle="--", linewidth=1)
    plt.text(-0.4, y, f" p={p_thr}", va="bottom")

# Annotate each bar with its original p-value.
for i, (val, pval) in enumerate(zip(res_df["neglog10p"], res_df["p"])):
    plt.text(i, val, f"{pval:.2e}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

# --- Figure 2: Plot effect sizes to show the magnitude of group differences. ---
plt.figure(figsize=(8, 4.5))
plt.bar(res_df["step"], res_df["eps2"])
plt.ylabel(r"Effect size $\varepsilon^2$ (approx.)")
plt.title("Effect size across four steps by level (ε², Kruskal–Wallis approximation)")
plt.xticks(rotation=0)

for i, v in enumerate(res_df["eps2"]):
    if np.isfinite(v):
        plt.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

res_df


## Step 10: Print Kruskal-Wallis P-Values

Print the four group-comparison p-values with readable labels for quick review.


In [ ]:
print("writing down emotionsp =", p_write)
print("imagine composition p =", p_imagine)
print("reference to picture p =", p_ref)
print("color expression p =", p_color)


## Step 11: Run Dunn's Post Hoc Test

Apply a Holm-adjusted Dunn test to determine which drawing-skill groups differ for the imagination-composition rating.


In [ ]:
!pip -q install scikit-posthocs
import scikit_posthocs as sp

# Use imagination and composition as the example; replace it with write, ref, or color if needed.
tmp = use[[level, imagine]].dropna()
sp.posthoc_dunn(tmp, val_col=imagine, group_col=level, p_adjust="holm")


## Step 12: Plot Imagination Ratings by Skill Level

Combine a box plot and jittered observations to show the distribution of imagination-composition ratings in each skill group.


In [ ]:
import matplotlib.pyplot as plt

# Prepare the three skill groups.
g1 = use.loc[use[level]==1, imagine].dropna()
g2 = use.loc[use[level]==2, imagine].dropna()
g3 = use.loc[use[level]==3, imagine].dropna()

plt.figure(figsize=(6,4))

# Draw the box plot.
plt.boxplot([g1, g2, g3], tick_labels=["Level 1","Level 2","Level 3"], showfliers=False,
            medianprops=dict(color="purple", linewidth=3))

# Add jittered observations so overlapping points remain visible.
import numpy as np
for i, g in enumerate([g1, g2, g3], start=1):
    x = np.random.normal(i, 0.04, size=len(g))
    plt.scatter(x, g, alpha=0.25)

plt.ylabel("Importance rating: Imagine composition (1–5)")
plt.title("Imagine composition importance by art level")
plt.tight_layout()
plt.show()


## Step 13: Plot the Other Drawing-Step Ratings

Create box plots with jittered observations for writing emotions, using reference images, and color expression.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Plot the other three steps; add `imagine` here if desired.
plots = [
    ("Writing down emotions", write),
    ("Reference to picture", ref),
    ("Color expression", color),
]

for title, col in plots:
    g1 = use.loc[use[level]==1, col].dropna()
    g2 = use.loc[use[level]==2, col].dropna()
    g3 = use.loc[use[level]==3, col].dropna()

    plt.figure(figsize=(6,4))

    # Use tick_labels to avoid the deprecated box-plot label argument.
    plt.boxplot([g1, g2, g3],
                tick_labels=["Level 1","Level 2","Level 3"],
                showfliers=False,
               medianprops=dict(color="red", linewidth=3))

    # jitter points
    for i, g in enumerate([g1, g2, g3], start=1):
        x = np.random.normal(i, 0.04, size=len(g))
        plt.scatter(x, g, alpha=0.25)

    plt.ylabel("Importance rating (1–5)")
    plt.title(f"{title} importance by art level")
    plt.tight_layout()
    plt.show()


## Step 14: Create an Alternate Imagination Plot

Generate a second imagination-composition plot with a different median highlight for presentation comparison.


In [ ]:
import matplotlib.pyplot as plt

# Prepare the three skill groups.
g1 = use.loc[use[level]==1, imagine].dropna()
g2 = use.loc[use[level]==2, imagine].dropna()
g3 = use.loc[use[level]==3, imagine].dropna()

plt.figure(figsize=(6,4))

# Draw the box plot.
plt.boxplot([g1, g2, g3], tick_labels=["Level 1","Level 2","Level 3"], showfliers=False,
            medianprops=dict(color="red", linewidth=3))

# Add jittered observations so overlapping points remain visible.
import numpy as np
for i, g in enumerate([g1, g2, g3], start=1):
    x = np.random.normal(i, 0.04, size=len(g))
    plt.scatter(x, g, alpha=0.25)

plt.ylabel("Importance rating: Imagine composition (1–5)")
plt.title("Imagine composition importance by art level")
plt.tight_layout()
plt.show()


## Step 15: Visualize All Step Distributions

Overlay violin plots, box plots, and jittered observations to compare the full distribution of every drawing-step rating across skill levels.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plots = [
    ("Writing down emotions", write),
    ("Imagine composition", imagine),
    ("Reference to picture", ref),
    ("Color expression", color),
]

for title, col in plots:
    g1 = use.loc[use[level]==1, col].dropna()
    g2 = use.loc[use[level]==2, col].dropna()
    g3 = use.loc[use[level]==3, col].dropna()

    data = [g1, g2, g3]

    plt.figure(figsize=(6,4))

    # 1) The violin shows distribution shape; wider areas contain more observations.
    plt.violinplot(data, showmeans=False, showmedians=False, showextrema=False)

    # 2) The box plot shows the median and interquartile range.
    plt.boxplot(data, tick_labels=["Level 1","Level 2","Level 3"],
                showfliers=False, widths=0.25,
                medianprops=dict(color="red", linewidth=3)
                )

    # 3) Jittered points show the individual observations.
    for i, g in enumerate(data, start=1):
        x = np.random.normal(i, 0.04, size=len(g))
        plt.scatter(x, g, alpha=0.20, s=10)

    plt.title(f"{title} importance by art level")
    plt.ylabel("Importance rating (1–5)")
    plt.tight_layout()
    plt.show()


## Step 16: Define Emotion Outcome Aliases

Assign short aliases for negative-release and positive-emotion outcomes and preview the prepared group-analysis table.


In [ ]:
neg, pos = "neg", "pos"

use

## Step 17: Calculate Level 1 Correlations

Compute Spearman correlations between each drawing-step rating and both emotion outcomes for drawing-skill level 1.


In [ ]:
from scipy.stats import spearmanr
u1 = use[use[level]==1][[write, imagine, ref, color, neg, pos]].dropna()
print("Level 1 n =", len(u1))
print("write vs neg:", spearmanr(u1[write], u1[neg]))
print("write vs pos:", spearmanr(u1[write], u1[pos]))
print("imagine vs neg:", spearmanr(u1[imagine], u1[neg]))
print("imagine vs pos:", spearmanr(u1[imagine], u1[pos]))
print("ref vs neg:", spearmanr(u1[ref], u1[neg]))
print("ref vs pos:", spearmanr(u1[ref], u1[pos]))
print("color vs neg:", spearmanr(u1[color], u1[neg]))
print("color vs pos:", spearmanr(u1[color], u1[pos]))


## Step 18: Calculate Level 2 Correlations

Compute the same step-to-outcome Spearman correlations for drawing-skill level 2.


In [ ]:
u2 = use[use[level]==2][[write, imagine, ref, color, neg, pos]].dropna()

print("Level 2 n =", len(u2))
print("write vs neg:", spearmanr(u2[write], u2[neg]))
print("write vs pos:", spearmanr(u2[write], u2[pos]))
print("imagine vs neg:", spearmanr(u2[imagine], u2[neg]))
print("imagine vs pos:", spearmanr(u2[imagine], u2[pos]))
print("ref vs neg:", spearmanr(u2[ref], u2[neg]))
print("ref vs pos:", spearmanr(u2[ref], u2[pos]))
print("color vs neg:", spearmanr(u2[color], u2[neg]))
print("color vs pos:", spearmanr(u2[color], u2[pos]))


## Step 19: Calculate Level 3 Correlations

Compute the same step-to-outcome Spearman correlations for drawing-skill level 3.


In [ ]:
u3 = use[use[level]==3][[write, imagine, ref, color, neg, pos]].dropna()

print("Level 3 n =", len(u3))
print("write vs neg:", spearmanr(u3[write], u3[neg]))
print("write vs pos:", spearmanr(u3[write], u3[pos]))
print("imagine vs neg:", spearmanr(u3[imagine], u3[neg]))
print("imagine vs pos:", spearmanr(u3[imagine], u3[pos]))
print("ref vs neg:", spearmanr(u3[ref], u3[neg]))
print("ref vs pos:", spearmanr(u3[ref], u3[pos]))
print("color vs neg:", spearmanr(u3[color], u3[neg]))
print("color vs pos:", spearmanr(u3[color], u3[pos]))


## Step 20: Summarize Selected Correlations

Collect selected writing and reference-image correlations into a compact table for comparison across skill levels.


In [ ]:
def one_row(u, step, outcome):
    r, p = spearmanr(u[step], u[outcome])
    return round(r, 3), p

rows = []

for lvl, u in [(1,u1),(2,u2),(3,u3)]:
    r1,p1 = one_row(u, write, neg)
    r2,p2 = one_row(u, write, pos)
    r3,p3 = one_row(u, ref, neg)
    r4,p4 = one_row(u, ref, pos)

    rows.append([lvl, len(u), r1, p1, r2, p2, r3, p3, r4, p4])

table = pd.DataFrame(rows, columns=[
    "level","n",
    "write~neg_r","write~neg_p",
    "write~pos_r","write~pos_p",
    "ref~neg_r","ref~neg_p",
    "ref~pos_r","ref~pos_p"
])
table


## Step 21: Build the Complete Correlation Table

Calculate every drawing-step-to-emotion correlation using a consistent complete-case sample within each skill level.


In [ ]:
from scipy.stats import spearmanr
import pandas as pd

# These aliases should already be defined above; otherwise, set them to match the `use` table.
# level = use.columns[0]
# write, imagine, ref, color = use.columns[1], use.columns[2], use.columns[3], use.columns[4]
# neg = "neg"
# pos = "pos"

steps = {
    "write": write,
    "imagine": imagine,
    "ref": ref,
    "color": color
}

outcomes = {
    "neg": neg,
    "pos": pos
}

rows = []

for lvl in [1, 2, 3]:
    # Use the same complete-case sample for every correlation within a skill group.
    cols_need = list(steps.values()) + list(outcomes.values())
    u = use[use[level] == lvl][cols_need].dropna()

    row = {"level": lvl, "n": len(u)}

    for s_key, s_col in steps.items():
        for o_key, o_col in outcomes.items():
            r, p = spearmanr(u[s_col], u[o_col])
            row[f"{s_key}~{o_key}_r"] = round(r, 3)
            row[f"{s_key}~{o_key}_p"] = p

    rows.append(row)

table = pd.DataFrame(rows)
table


## Step 22: Train a Gradient Boosting Model

Evaluate a histogram gradient boosting regressor with five-fold cross-validation and estimate predictor importance by permutation.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

# ===== Prepare data using the current Dataset schema. =====

hgb_df = df_raw.copy()
hgb_df = hgb_df.replace({"na": np.nan, "NA": np.nan, "N/A": np.nan, "": np.nan, " ": np.nan})

for c in hgb_df.columns:
    if c not in {"text", "painting_path", "processed_painting_path"}:
        hgb_df[c] = pd.to_numeric(hgb_df[c], errors="coerce")

y_col = "positive_emotion"
drop_cols = [
    "student_id", "text", "continue_participation", "painting_path",
    "processed_painting_path", "negative_release", "others",
]

hgb_df = hgb_df.dropna(subset=[y_col])
y = hgb_df[y_col].astype(float)
X = hgb_df.drop(columns=[y_col] + drop_cols, errors="ignore")
X = X.select_dtypes(include=[np.number])
X = X.dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=True) > 1]

# ===== Define evaluation metrics. =====
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))
rmse_scorer = make_scorer(rmse, greater_is_better=False)

# ===== Fit a HistGradientBoostingRegressor, which can handle NaN values. =====
hgb = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=3,
    max_iter=400,
    random_state=42
)

# ===== Evaluate predictive performance with five-fold cross-validation. =====
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(
    hgb, X, y, cv=cv,
    scoring={"r2":"r2","neg_mae":"neg_mean_absolute_error","neg_rmse":rmse_scorer}
)

print("HGB CV R2   :", scores["test_r2"].mean(), "+/-", scores["test_r2"].std())
print("HGB CV MAE  :", -scores["test_neg_mae"].mean(), "+/-", scores["test_neg_mae"].std())
print("HGB CV RMSE :", -scores["test_neg_rmse"].mean(), "+/-", scores["test_neg_rmse"].std())

# ===== Estimate feature importance with permutation importance. =====
# This example uses one train/test split; a stricter analysis would average importance across folds.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
hgb.fit(Xtr, ytr)

perm = permutation_importance(
    hgb, Xte, yte,
    n_repeats=30,
    random_state=42,
    scoring="r2"
)

pi = pd.DataFrame({
    "feature": X.columns,
    "perm_r2_drop_mean": perm.importances_mean,
    "perm_r2_drop_std": perm.importances_std
}).sort_values("perm_r2_drop_mean", ascending=False)

print("\nPermutation importance (bigger = more important):")
print(pi.to_string(index=False))


## Step 23: Run Hierarchical Regression

Compare nested regression models, test incremental explained variance, and report standardized coefficients for the full model.


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# 1) Create an analysis copy of the validated source data.
hier_df = df_raw.copy()
hier_df = hier_df.replace({"na": np.nan, "NA": np.nan, "N/A": np.nan, "": np.nan, " ": np.nan})
hier_df = hier_df.dropna(how="all")

# 2) Convert all columns except text and image paths to numeric values where possible.
for c in hier_df.columns:
    if c not in {"text", "painting_path", "processed_painting_path"}:
        hier_df[c] = pd.to_numeric(hier_df[c], errors="coerce")

# 3) Define the outcome and the predictor blocks; excluded columns do not enter the models.
y_col = "positive_emotion"
drop_cols = ["student_id", "text", "continue_participation", "negative_release", "others"]

block1 = ["drawing_skills"]
block3 = ["write_emotions", "imagine_composition", "examples", "color_expression", "level_difficulty"]
block2 = ["drawing_participation", "enjoy_drawing"]

# 4) Use complete cases for every full-model variable so Delta R-squared comparisons use the same sample.
all_predictors = block1 + block2 + block3
data = hier_df[[y_col] + all_predictors].dropna().copy()
y = y_col

# 5) Fit the nested hierarchical models.
f1 = f"{y} ~ " + " + ".join(block1)
f2 = f"{y} ~ " + " + ".join(block1 + block2)
f3 = f"{y} ~ " + " + ".join(block1 + block2 + block3)

m1 = smf.ols(f1, data=data).fit()
m2 = smf.ols(f2, data=data).fit()
m3 = smf.ols(f3, data=data).fit()

# 6) Compare R-squared and incremental R-squared.
comp = pd.DataFrame({
    "Model": ["M1: drawing skills", "M2: + participation + enjoyment", "M3: + drawing-step factors"],
    "N": [int(m1.nobs), int(m2.nobs), int(m3.nobs)],
    "R2": [m1.rsquared, m2.rsquared, m3.rsquared],
    "Adj_R2": [m1.rsquared_adj, m2.rsquared_adj, m3.rsquared_adj],
})
comp["Delta_R2"] = comp["R2"].diff()
print(comp)

# 7) Test incremental significance with nested-model F tests.
print("\nF-change tests (nested model comparison):")
print(anova_lm(m1, m2, m3))

# 8) Report standardized coefficients from the full model for easier comparison.
cols_to_z = [y] + all_predictors
z = data[cols_to_z].apply(lambda s: (s - s.mean())/s.std(ddof=0))
m3_std = smf.ols(f3, data=z).fit()

std_table = pd.DataFrame({
    "term": m3_std.params.index,
    "beta_std": m3_std.params.values,
    "p_value": m3_std.pvalues.values,
    "ci_low": m3_std.conf_int()[0].values,
    "ci_high": m3_std.conf_int()[1].values
}).query("term != 'Intercept'").sort_values("beta_std", ascending=False)

print("\nStandardized betas (Model 3):")
print(std_table.to_string(index=False))


## Step 24: Evaluate Preference-Effectiveness Alignment

Estimate drawing-step effectiveness weights, calculate participant alignment scores, test their relationship with positive emotion, and visualize the result.


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

alignment_df = df_raw.copy()
for c in ["positive_emotion", "write_emotions", "imagine_composition", "examples",
          "color_expression", "drawing_skills", "level_difficulty",
          "drawing_participation", "enjoy_drawing"]:
    alignment_df[c] = pd.to_numeric(alignment_df[c], errors="coerce")

# Four drawing-step variables used throughout the notebook.
steps = ["write_emotions", "imagine_composition", "examples", "color_expression"]

# Optional controls, consistent with the hierarchical regression.
controls = ["drawing_skills", "level_difficulty", "drawing_participation", "enjoy_drawing"]

# Keep required columns and complete cases so every model uses the same sample.
need_cols = ["positive_emotion"] + steps + controls
data = alignment_df[need_cols].dropna().copy()

print("N used for RQ4:", len(data))

# =========================
# 1) Estimate the effectiveness-weight vector E.
#    Use standardized regression coefficients as the effectiveness weight of each step.
#    Option A uses only the steps; option B adds controls to estimate adjusted associations.
# =========================

def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

# Select option A or B; option B estimates each step's association after adjustment.
USE_CONTROLS_FOR_WEIGHTS = True

if USE_CONTROLS_FOR_WEIGHTS:
    cols_for_weights = ["positive_emotion"] + steps + controls
    formula = "positive_emotion ~ " + " + ".join(steps + controls)
else:
    cols_for_weights = ["positive_emotion"] + steps
    formula = "positive_emotion ~ " + " + ".join(steps)

z = data[cols_for_weights].apply(zscore)

m = smf.ols(formula, data=z).fit()
beta = m.params[steps]  # Keep the standardized coefficients for the four steps only.
print("\nStandardized betas (effectiveness weights):")
print(beta)

# Convert beta into a non-negative weight vector that emphasizes positive associations.
# Remove the clipping step if negative weights should act as penalties.
beta_pos = beta.clip(lower=0)

# Guard against division by zero if every coefficient is non-positive.
if beta_pos.sum() == 0:
    raise ValueError("All step betas are <= 0; cannot form positive effectiveness weights.")
E = (beta_pos / beta_pos.sum()).values  # Normalize the weights so they sum to one.
E_names = steps

print("\nNormalized effectiveness weights E (sum=1):")
for name, w in zip(E_names, E):
    print(f"{name}: {w:.4f}")

# =========================
# 2) Calculate each participant's preference-weight vector w_i.
#    w_i = step_score / sum(step_scores)
# =========================
S = data[steps].copy()

row_sum = S.sum(axis=1)
# Guard against participants whose four step scores sum to zero.
row_sum = row_sum.replace(0, np.nan)

W = S.div(row_sum, axis=0)  # Normalize each participant's row.
W = W.fillna(0)

# =========================
# 3) Calculate preference-effectiveness alignment.
#    Two candidate alignment measures are calculated below.
#    A) Weighted effectiveness index: alignment = Σ w_i * E
#    B) Cosine similarity: cos(w_i, E)
# =========================

# A) Weighted effectiveness-preference index.
data["alignment_index"] = W.values @ E

# B) Cosine similarity, ranging from 0 to 1 with larger values indicating closer alignment.
E_vec = E / np.linalg.norm(E)
W_norm = np.linalg.norm(W.values, axis=1)
# Guard against division by zero.
W_norm = np.where(W_norm == 0, 1e-12, W_norm)
data["alignment_cosine"] = (W.values @ E_vec) / W_norm

print("\nAlignment summary:")
print(data[["alignment_index", "alignment_cosine"]].describe())

# =========================
# 4) Use regression to test whether greater alignment is associated with positive emotion.
#    Compare an unadjusted model with a model that includes the control variables.
# =========================

# Select the alignment measure used in the regression models.
ALIGN_COL = "alignment_index"   # Alternatively, use "alignment_cosine".

# (1) Unadjusted regression.
r1 = smf.ols(f"positive_emotion ~ {ALIGN_COL}", data=data).fit()
print("\n[Model R1] positive_emotion~ alignment")
print(r1.summary().tables[1])

# (2) Adjusted regression, preferred for reporting.
r2 = smf.ols(f"positive_emotion ~ {ALIGN_COL} + " + " + ".join(controls), data=data).fit()
print("\n[Model R2] positive_emotion ~ alignment + controls")
print(r2.summary().tables[1])

# Focus on the coefficient and p-value for ALIGN_COL.
# A significant positive coefficient supports a positive alignment-emotion association.
# A non-significant coefficient indicates insufficient evidence or a weak association.

# =========================
# 5) Optionally compare group-level preference rankings with the effectiveness ranking.
# =========================
group_means = data.groupby("drawing_skills")[steps].mean()

# Rank effectiveness by beta_pos; use beta instead if negative associations should be retained.
eff_rank = beta_pos.sort_values(ascending=False).index.tolist()

print("\nEffectiveness rank (by beta_pos):", eff_rank)

print("\nGroup-level preference ranks vs effectiveness rank:")
for lvl, row in group_means.iterrows():
    pref_rank = row.sort_values(ascending=False).index.tolist()
    # Convert rankings to ordinal positions and compare them with Spearman correlation.
    eff_order = {name: i for i, name in enumerate(eff_rank)}
    pref_order = {name: i for i, name in enumerate(pref_rank)}
    x = [eff_order[s] for s in steps]
    y = [pref_order[s] for s in steps]
    rho, p = spearmanr(x, y)
    print(f"drawing_skills={lvl}: preference rank={pref_rank} | Spearman rho={rho:.3f}, p={p:.3f}")

# =========================
# 6) Visualize alignment versus positive emotion, colored by drawing-skill level.
# =========================
plt.figure(figsize=(7,5))
for lvl, sub in data.groupby("drawing_skills"):
    plt.scatter(sub[ALIGN_COL], sub["positive_emotion"], s=12, alpha=0.55, label=f"ability={int(lvl)}")
plt.xlabel(ALIGN_COL)
plt.ylabel("positive_emotion")
plt.title("Alignment vs Positive Mood (colored by art ability)")
plt.legend()
plt.tight_layout()
plt.show()

## Step 25: Cross-Fit the Alignment Analysis

Estimate effectiveness weights out of sample, calculate cross-fitted alignment scores, test adjusted associations, and examine weight stability across folds.


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import KFold
from scipy.stats import zscore


crossfit_df = df_raw.copy()
crossfit_df = crossfit_df.replace({"na": np.nan, "NA": np.nan, "N/A": np.nan, "": np.nan, " ": np.nan})
crossfit_df = crossfit_df.dropna(how="all")

for c in crossfit_df.columns:
    if c not in {"text", "painting_path", "processed_painting_path"}:
        crossfit_df[c] = pd.to_numeric(crossfit_df[c], errors="coerce")

target = "positive_emotion"
steps = ["write_emotions", "imagine_composition", "examples", "color_expression"]
controls = ["drawing_skills", "level_difficulty", "drawing_participation", "enjoy_drawing"]

need_cols = [target] + steps + controls
data = crossfit_df[need_cols].dropna().reset_index(drop=True).copy()
print("N used:", len(data))

# ========= 1) Cross-fitting: estimate E on each training fold and calculate alignment on its test fold. =========
kf = KFold(n_splits=5, shuffle=True, random_state=42)

alignment_oof = np.full(len(data), np.nan)  # out-of-fold alignment_index
E_oof = []  # Store fold-specific E vectors for the stability summary.

USE_CONTROLS_FOR_WEIGHTS = True  # True uses steps plus controls; False uses the steps only.

for fold, (tr_idx, te_idx) in enumerate(kf.split(data), 1):
    train = data.loc[tr_idx].copy()
    test  = data.loc[te_idx].copy()

    # --- 1.1 Estimate effectiveness betas from standardized training-fold data. ---
    if USE_CONTROLS_FOR_WEIGHTS:
        cols_for_weights = [target] + steps + controls
        X_vars = steps + controls
    else:
        cols_for_weights = [target] + steps
        X_vars = steps

    ztr = train[cols_for_weights].apply(lambda s: (s - s.mean()) / s.std(ddof=0))
    formula = f"{target} ~ " + " + ".join(X_vars)
    m = smf.ols(formula, data=ztr).fit()

    beta = m.params[steps]
    beta_pos = beta.clip(lower=0)
    if beta_pos.sum() == 0:
        # Fall back to equal weights if every coefficient is non-positive.
        E = np.ones(len(steps)) / len(steps)
    else:
        E = (beta_pos / beta_pos.sum()).values

    E_oof.append(E)

    # --- 1.2 Calculate test-fold preference weights and alignment_index = w_i dot E. ---
    S = test[steps].copy()
    row_sum = S.sum(axis=1).replace(0, np.nan)
    W = S.div(row_sum, axis=0).fillna(0)

    alignment = W.values @ E
    alignment_oof[te_idx] = alignment

    print(f"Fold {fold}: E = {dict(zip(steps, np.round(E,4)))}")

data["alignment_cf"] = alignment_oof

# ========= 2) Regress the outcome on the out-of-fold alignment_cf values. =========
# (R1) Alignment only.
r1 = smf.ols(f"{target} ~ alignment_cf", data=data).fit()

# (R2) alignment + controls
r2 = smf.ols(f"{target} ~ alignment_cf + " + " + ".join(controls), data=data).fit()

print(f"\n[Cross-fitted R1] {target} ~ alignment_cf")
print(r1.summary().tables[1])

print(f"\n[Cross-fitted R2] {target} ~ alignment_cf + controls")
print(r2.summary().tables[1])

# ========= 3) Summarize the stability of E across folds. =========
E_df = pd.DataFrame(E_oof, columns=steps)
print("\nE stability across folds (mean ± std):")
print(pd.DataFrame({"mean": E_df.mean(), "std": E_df.std()}))

# ========= 4) Visualize cross-fitted alignment versus positive emotion. =========
import matplotlib.pyplot as plt
plt.figure(figsize=(7,5))
plt.scatter(data["alignment_cf"], data[target], s=12, alpha=0.6)
plt.xlabel("alignment_cf (cross-fitted)")
plt.ylabel(target)
plt.title("Cross-fitted Alignment vs Positive Mood")
plt.tight_layout()
plt.show()
